# Overview
This notebook prepares a sample of agent skills from the GitSkills dataset for automated security analysis using MalSkills.

# Scope
For initial testing, we will select 100 randomly ordered SKILL.md records from the GitSkills database. This smaller sampler will be used to develop and validate the automated analysis workflow before expanding to a larger dataset.

# Sampling Approach
For this initial experiment, I am randomly selecting 100 SKILL.md files. The SQL query filters for records containing a SKILL.md file with non-null content. The selected files are then exported into indiviudally numbered folders so they can be processed by MalSkills.

For this stage, I will only be exporting SKILL.md files, and not the additional files that may be included in the skill directory.

To keep track of the exported skills, a CSV file (skill_mapping.csv) will be generated. This file maps eaech numbered skill folder to its original GitSkills repo and file path, allowing us to trace any findings from MalSkills back to their original source.

## Connect

Connect to the SQL database prior to running the export script.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)  # max characters shown per cell in tables; raise it to see longer text

# Works whether VS Code runs the notebook from notebooks/ or from the project root
ROOT = Path.cwd() if (Path.cwd() / "data" / "sample").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} (see README)"

# Read-only, so nothing in the database can be changed by accident
conn = sqlite3.connect(f"{DB_PATH.resolve().as_uri()}?mode=ro", uri=True)


# sql("SELECT ...") -> DataFrame. Use ? placeholders for values: sql("... WHERE name = ?", ("pdf",))
def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)


print("Connected to", DB_PATH.relative_to(ROOT))

Connected to data\sample\agent_skills_sample.db


In [2]:
query = "SELECT COUNT(*) FROM artifacts"

total_skills = conn.execute(query).fetchone()[0]

print(f"Total skills: {total_skills}")

Total skills: 29786


In [3]:
query = "SELECT COUNT(*) FROM artifact_siblings"

total_sibling = conn.execute(query).fetchone()[0]

print(f"Total sibling entries: {total_sibling}")

Total sibling entries: 47829


In [4]:
query = """
SELECT
    entry_type,
    COUNT(*) AS total,
    SUM(CASE WHEN content IS NOT NULL THEN 1 ELSE 0 END) AS with_content
FROM artifact_siblings
GROUP BY entry_type
"""

sql(query)

,entry_type,total,with_content
0,dir,8783,0
1,file,39046,22278


In [5]:
query = """
SELECT
    COUNT(DISTINCT repo_full_name || '|' || artifact_path)
        AS skills_with_siblings,
    COUNT(DISTINCT CASE
        WHEN entry_type = 'file' AND content IS NOT NULL
        THEN repo_full_name || '|' || artifact_path
    END) AS skills_with_file_content
FROM artifact_siblings
"""

sql(query)

,skills_with_siblings,skills_with_file_content
0,4464,4171


In [6]:
query = """
SELECT
    repo_full_name,
    artifact_path,
    COUNT(*) AS total_files,
    SUM(CASE WHEN content IS NOT NULL THEN 1 ELSE 0 END) AS available_files
FROM artifact_siblings
WHERE entry_type = 'file'
GROUP BY repo_full_name, artifact_path
"""

coverage = sql(query)

coverage.head()

,repo_full_name,artifact_path,total_files,available_files
0,012e/thesis,.agents/skills/runtime/SKILL.md,5,5
1,0disoft/mustflow,.mustflow/skills/interpreter-engineering-review/SKILL.md,4,4
2,0ui-labs/GSD-CC,gsd-cc/skills/seed/SKILL.md,20,15
3,0x0xfan/videocut-skills-2,.agent/skills/videocut-自更新/SKILL.md,1,1
4,0x2kNJ/cpo,SKILL.md,11,9


## Export Script

In [10]:
# Export multiple skills for linting
from pathlib import Path

# pick 100 random skills from GitSkills

skills_to_export = sql("""
    SELECT repo_full_name, path, content
    FROM artifacts
    WHERE path LIKE '%/SKILL.md'
        AND content IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 100
""") # Change LIMIT as needed

# export each skill into its own folder
for index, row in skills_to_export.iterrows(): # iterrows(): loop through each row of DataFrame
    # extract repo name, original path, and file contents.
    repo = row["repo_full_name"]
    path = row["path"]
    content = row["content"]
    siblings = sql("""
        SELECT entry_name, entry_type, content
        FROM artifact_siblings
        WHERE repo_full_name = ?
            AND artifact_path = ?
            AND entry_type = 'file'
    """, (repo, path))

    # unique numbered folder per skill up to 4 numbers.
    path_name = f"skill_{index:04d}"

    # each skill gets its own folder inside exported_skills
    output = Path(f"../exported_skills/{path_name}/SKILL.md")
    output.parent.mkdir(parents=True, exist_ok=True) # create folder if n/a
    output.write_text(content, encoding="utf-8")

    # export available sibling files
    skill_folder = output.parent.resolve()

    for _, sibling in siblings.iterrows():
        if pd.isna(sibling["content"]):
            continue

        sibling_path = Path(sibling["entry_name"])

        # ensure the sibling stays inside skill folder
        if sibling_path.is_absolute() or ".." in sibling_path.parts:
            continue

        sibling_output = (skill_folder / sibling_path).resolve()

        if not sibling_output.is_relative_to(skill_folder):
            continue

        if sibling_output == output.resolve():
            continue

        sibling_output.parent.mkdir(parents=True, exist_ok=True)
        sibling_output.write_text(sibling["content"], encoding="utf-8")

    print (f"{path_name}: {repo}/{path}")

print(f"\nExported {len(skills_to_export)} skills.")

# Save a mapping between exported folders and original GitSkills records
skill_mapping = skills_to_export[["repo_full_name", "path"]].copy()

# Generate folder names matching those created during export
skill_mapping["skill_id"] = [f"skill_{index:04d}" for index in skills_to_export.index]

# Organize columns
skill_mapping = skill_mapping[["skill_id", "repo_full_name", "path"]]

# Save mapping to CSV for later analysis
mapping_path = Path("../exported_skills/skill_mapping.csv")
skill_mapping.to_csv(mapping_path, index=False)
print(f"Saved skill mapping to: {mapping_path}")

skill_0000: Isawa-Mac/360/.cursor/skills/theme-wind-gradient-background/SKILL.md
skill_0001: nicholls-inc/xylem/.claude/skills/xylem-drain/SKILL.md
skill_0002: qhjqhj00/research-skills-pool/skill-factory/output/fschlatt-ner-eval/SKILL.md
skill_0003: bottle-note/bottle-note-api-server/.claude/skills/debug/SKILL.md
skill_0004: modbender/skill-library-mcp/data/homepod-tts/SKILL.md
skill_0005: HolobiomicsLab/asb-skill-collections/collections/metabolomics/v2/skills/molecular-formula-assignment/SKILL.md
skill_0006: micsapp/micstec-skills/claude-code/plugins-plus-skills/plugins/testing/load-balancer-tester/skills/testing-load-balancers/SKILL.md
skill_0007: tools-only/X-Skills/development/web/053-application-profiling_db873957/skill.md
skill_0008: marktantongco/opencodelinux/skills/grill-me/SKILL.md
skill_0009: ricardonevesbraga/flowgrammers-skills/engineering/agenthub/skills/init/SKILL.md
skill_0010: orieken/saturday-monorepo/.claude/skills/scaffold-docs/SKILL.md
skill_0011: MikeCheng1208/Bat

## Verify the exported files

In [8]:
from pathlib import Path

export_dir = Path("../exported_skills")

# Find all SKILL.md files inside the exported skill folders
exported_files = list(export_dir.glob("skill_*/SKILL.md"))

print(f"Expected files: 100")
print(f"Actual files: {len(exported_files)}")

Expected files: 100
Actual files: 1000
